# 00 · Configuración del entorno

Fase 0 del proyecto. Prepara el entorno en Google Colab, fija la semilla, descarga el dataset y verifica que coincida con la documentación de Realinho et al. (2022).

In [ ]:
# Clonar el repositorio (solo en Colab). Cambiar REPO_URL por la URL real.
import os, sys
REPO_URL = "https://github.com/USUARIO/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"

if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Directorio de trabajo:", os.getcwd())

In [ ]:
# Instalar dependencias con versiones fijadas
%pip install -q -r requirements.txt

Si Colab pide reiniciar la sesión después de instalar, reiniciar y continuar desde la siguiente celda (ejecutando antes la primera).

In [ ]:
from src import config
from src.utils import set_seed, environment_report

set_seed()
env = environment_report()
env

In [ ]:
# Guardar el reporte del entorno para el anexo de reproducibilidad
import json
config.FIGURES.parent.mkdir(parents=True, exist_ok=True)
with open(config.FIGURES.parent / "entorno.json", "w", encoding="utf-8") as f:
    json.dump(env, f, indent=2, ensure_ascii=False)

## Descarga y verificación del dataset

In [ ]:
from src.data import load_raw, binarize_target

df = load_raw()
print("Registros:", df.shape[0], "| Columnas:", df.shape[1])

In [ ]:
# Verificaciones contra la documentación publicada
print("Valores faltantes:", int(df.isna().sum().sum()))
print("Duplicados:", int(df.duplicated().sum()))
print("Predictoras:", df.shape[1] - 1)
print()
print(df[config.TARGET_COL].value_counts())
print()
print((df[config.TARGET_COL].value_counts(normalize=True) * 100).round(1))

Valores esperados según Realinho et al. (2022): 4.424 registros, sin valores faltantes, Graduate 2.209, Dropout 1.421, Enrolled 794. El número de predictoras resuelve la diferencia 35 frente a 36 señalada en la revisión.

In [ ]:
df = binarize_target(df)
print(df[config.BINARY_TARGET].value_counts())
print("Proporción de deserción:", round(df[config.BINARY_TARGET].mean() * 100, 1), "%")

In [ ]:
from src.data import features_for_moment
for m in config.MOMENTS:
    print(f"{m}: {len(features_for_moment(df, m))} variables")